# Thulla DMC — Colab T4 training

Minimal DouZero-style self-play for a decent 4-player Thulla bot.

Learns **card plays** and **ASK/PASS** on the take phase (victims always give).

Obs include known holdings (thulla/take) and the **free unknown** card pool; heads-up uses deduced opponent hands.

**Setup:** Runtime → Change runtime type → **T4 GPU** (not CPU).

- Learner on **T4**; self-play actors on **CPU** (default 6)
- Latest checkpoint every ~3 min → `model.tar`
- **Eval vs random** every 15 min (50 games)
- **Eval vs heuristic** every 30 min (50 games) → saves `model_best.tar` when P(not last) improves
- Eval summaries append to **`eval_log.csv`** in the same Drive folder

**Note:** If you changed obs/encoding recently, start a **fresh** checkpoint folder or delete old `model.tar`.


## 1. Install deps

In [ ]:
%pip install -q "numpy>=1.24" "torch>=2.0"

## 2. Get thulla-ai code

Either clone your repo, or upload a zip of `thulla-ai` to Drive and set `REPO_DIR` below.

In [ ]:
import os
import sys

import torch

# --- edit these ---
REPO_URL = ""  # e.g. "https://github.com/YOU/thulla-ai.git" or leave blank if uploading
REPO_DIR = "/content/thulla-ai"
DRIVE_CKPT = "/content/drive/MyDrive/thulla_dmc_ckpts"
# ------------------

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CKPT, exist_ok=True)

if REPO_URL and not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
elif not os.path.isdir(REPO_DIR):
    raise SystemExit(
        f"Missing {REPO_DIR}. Set REPO_URL or upload thulla-ai there (must contain thulla/ and thulla_dmc/)."
    )

sys.path.insert(0, REPO_DIR)

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Runtime → Change runtime type → Hardware accelerator → T4 GPU, then re-run."
    )

print("REPO_DIR:", REPO_DIR)
print("checkpoints:", DRIVE_CKPT)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


## 3. Train on T4 (auto-resume if checkpoint exists)

Defaults: GPU learner, 6 CPU actors, batch 512, latest save every 3 min.

Timed evals (training pauses briefly): **random every 15 min**, **heuristic every 30 min** (50 games each). Best-vs-heuristic → `model_best.tar`.


In [ ]:
import os
from thulla_dmc.arguments import parse_args
from thulla_dmc.train import train

ckpt_dir = os.path.join(DRIVE_CKPT, "thulla_dmc")
has_ckpt = os.path.exists(os.path.join(ckpt_dir, "model.tar"))

# Colab T4 preset — bump --num_actors to 8 if the GPU looks idle
argv = [
    "--savedir", DRIVE_CKPT,
    "--xpid", "thulla_dmc",
    "--training_device", "0",
    "--require_gpu",
    "--num_actors", "6",
    "--batch_size", "512",
    "--save_interval", "3",              # latest model.tar
    "--eval_random_minutes", "15",       # vs RandomPlayer
    "--eval_heuristic_minutes", "30",    # vs ComputerPlayer → model_best.tar
    "--eval_games", "50",
    "--total_episodes", "30000",
    "--exp_epsilon", "0.05",
    "--log_interval", "25",
]
if has_ckpt:
    argv.append("--load_model")
    print("Resuming from", ckpt_dir)
else:
    print("Starting fresh →", ckpt_dir)

flags = parse_args(argv)
train(flags)


## 4. Manual evaluate

Use `model_best.tar` if present (best vs heuristic); otherwise `model.tar`.
Primary metric: **P(not last)**.


In [ ]:
import os
from thulla_dmc.evaluate import evaluate

best = f"{DRIVE_CKPT}/thulla_dmc/model_best.tar"
latest = f"{DRIVE_CKPT}/thulla_dmc/model.tar"
ckpt = best if os.path.exists(best) else latest
print("Evaluating:", ckpt)

print("\n--- vs random ---")
evaluate(ckpt, num_games=200, device="cpu", opponent="random")

print("\n--- vs heuristic ---")
evaluate(ckpt, num_games=100, device="cpu", opponent="heuristic")